# ComfyUI + FLUX.3 [klein] server for the picture trial

Pushed by `novel ktrial up`. Installs a pinned ComfyUI on one T4, downloads the INT8 FLUX.3 [klein] files (no prompt
upsampler), probes one text-only and one 2-reference picture, then serves ComfyUI through a tunnel behind basic auth
and prints `KTRIAL URL <url>`. It exits on a stop file uploaded by the local side, after the idle limit, or at the
hours cap: cheap checks come first, so a failure costs minutes, not a session.

In [ ]:
# Cheap checks first: GPU present, every input dataset present. Nothing is installed yet.
import json, os, shutil, subprocess, sys, time
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
GPUS = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
        if l.startswith("GPU ")]
assert GPUS, "no GPU: the kernel must run with a GPU accelerator (NvidiaTeslaT4)"
print("gpus", len(GPUS))

def _find_dataset(slug: str) -> Path:
    candidates = [Path('/kaggle/input') / slug,
                  *Path('/kaggle/input/datasets').glob(f'*/{slug}')]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input: attach it to the kernel")

def _secret(slug: str, name: str) -> str:
    p = _find_dataset(slug) / name
    if not p.exists():
        raise FileNotFoundError(f"dataset '{slug}' has no file {name}")
    v = p.read_text().strip()
    assert v, f"{slug}/{name} is empty"
    return v

SRC = _find_dataset('@DATASET@')
WORK = Path('/tmp/ktrial')          # not /kaggle/working: everything there becomes kernel output
WORK.mkdir(parents=True, exist_ok=True)
shutil.copy(SRC / 'kcomfy.py', WORK / 'kcomfy.py')
sys.path.insert(0, str(WORK))
import kcomfy

session = json.loads((SRC / 'session.json').read_text(encoding='utf-8'))
for k in ("user", "password"):
    assert session.get(k), f"session.json has no {k}"
TUNNEL = session.get("tunnel", "ngrok")
assert TUNNEL in ("ngrok", "cloudflared"), f"unknown tunnel {TUNNEL!r}"
HF_TOKEN = _secret('hf-token', 'HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
NGROK_TOKEN = _secret('ngrok-token', 'NGROK_AUTHTOKEN') if TUNNEL == "ngrok" else None
print("dataset", SRC, sorted(p.name for p in SRC.iterdir()))
print("session", {k: v for k, v in session.items() if k != "password"}, "tunnel", TUNNEL)

In [ ]:
# Pinned ComfyUI in /tmp. Its requirements leave Kaggle's torch alone ("torch" is unpinned).
COMFY = Path('/tmp/ComfyUI')
t0 = time.time()
if not (COMFY / 'main.py').exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", kcomfy.COMFY_TAG,
                    "https://github.com/comfyanonymous/ComfyUI", str(COMFY)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt")], check=True)
extra = ["huggingface_hub"] + (["pyngrok"] if TUNNEL == "ngrok" else [])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
print("installed in %.1f min" % ((time.time() - t0) / 60))
print((COMFY / "comfyui_version.py").read_text().strip().splitlines()[-1])

# The T4 is sm_75: a torch without it fails here, in seconds, before the 11 GB download.
CHECK = ("import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda); "
         "print('arch', torch.cuda.get_arch_list()); "
         "print('bf16 supported (maybe emulated)', torch.cuda.is_bf16_supported()); "
         "assert 'sm_75' in torch.cuda.get_arch_list(), 'torch build lacks sm_75'")
subprocess.run([sys.executable, "-c", CHECK], check=True)

In [ ]:
# The INT8 diffusion model, its text encoder and the VAE. The 9B prompt upsampler is NOT downloaded.
from huggingface_hub import hf_hub_download

MODELS = COMFY / 'models'
t0 = time.time()
for rel, size in kcomfy.MODEL_FILES:
    dest = MODELS / rel
    if dest.exists() and dest.stat().st_size == size:
        print(f"have {rel} {size / 1e9:.2f} GB")
        continue
    try:
        hf_hub_download(kcomfy.HF_REPO, rel, local_dir=str(MODELS), token=HF_TOKEN)
    except Exception as e:
        name = type(e).__name__
        if "Gated" in name or "401" in str(e) or "403" in str(e):
            raise RuntimeError(f"{kcomfy.HF_REPO} is gated: accept the FLUX Kommunity License on its Hugging Face "
                               f"page with the account behind the hf-token dataset ({name})") from None
        raise
    got = dest.stat().st_size
    print(f"got  {rel} {got / 1e9:.2f} GB")
    assert got == size, f"{rel}: {got} bytes, expected {size}"
print("downloads %.1f min" % ((time.time() - t0) / 60))
print(subprocess.run(["df", "-h", "/tmp"], capture_output=True, text=True).stdout)

In [ ]:
# ComfyUI on one GPU only: GPU 1 by default when there are two (GPU 0 is the one text-to-speech uses).
GPU = str(session.get("gpu", 1)) if len(GPUS) > 1 else "0"
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU)
ARGS = session.get("comfy_args", kcomfy.COMFY_LAUNCH_ARGS)
LOG = Path('/tmp/comfy.log')
log_f = open(LOG, "w")
comfy_proc = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
                               "--disable-auto-launch", *ARGS],
                              cwd=str(COMFY), env=env, stdout=log_f, stderr=subprocess.STDOUT)
print("ComfyUI pid", comfy_proc.pid, "CUDA_VISIBLE_DEVICES", GPU, "args", ARGS)

def log_tail(n=60):
    return "\n".join(LOG.read_text(errors="replace").splitlines()[-n:])

local = kcomfy.Comfy("http://127.0.0.1:8188", timeout=10, retries=0)
deadline = time.time() + 600
while True:
    if comfy_proc.poll() is not None:
        raise RuntimeError(f"ComfyUI exited with {comfy_proc.returncode}:\n{log_tail()}")
    try:
        stats = local.ping()
        break
    except kcomfy.ComfyError:
        if time.time() > deadline:
            raise RuntimeError(f"ComfyUI did not answer in 10 min:\n{log_tail()}")
        time.sleep(3)
print(json.dumps(stats, indent=1)[:1500])

In [ ]:
# Probe: a text-only picture, then a picture with two references (made text-only first), all 1376x768.
# Fails fast on a black picture or a reference run slower than MAX_SECONDS.
from PIL import Image, ImageStat
import io

MAX_SECONDS = 180
MIN_STD = 8.0
W, H = 1376, 768
PROBE = Path('/kaggle/working/probe')
PROBE.mkdir(parents=True, exist_ok=True)
local = kcomfy.Comfy("http://127.0.0.1:8188", timeout=60, retries=2)

def looks_blank(data: bytes) -> bool:
    img = Image.open(io.BytesIO(data)).convert("L")
    return max(ImageStat.Stat(img).stddev) < MIN_STD

def probe(name, prompt, seed, refs=()):
    data, took = local.generate(prompt, seed, W, H, list(refs), f"probe/{name}", timeout=1800)
    (PROBE / f"{name}.png").write_bytes(data)
    print(f"probe {name}: {took:.1f}s")
    assert not looks_blank(data), f"probe {name}: the picture is blank (std below {MIN_STD}); see {PROBE}"
    return data, took

STYLE = "Detailed digital painting, soft natural light."
probe("text", f"{STYLE} A lighthouse on a rocky coast at dusk, waves breaking below, gulls in the sky.", 1)
person, _ = probe("ref_person", f"{STYLE} Head and shoulders portrait of a young man with short black hair, "
                  "a dark green wool coat and a grey scarf, plain light background.", 2)
room, _ = probe("ref_room", f"{STYLE} An empty old library with tall wooden shelves, a round table, "
                "warm lamp light, no people.", 3)
ref_names = [local.upload_bytes(person, "probe_ref_person.png"), local.upload_bytes(room, "probe_ref_room.png")]
_, took = probe("two_refs", f"{STYLE} The man from {kcomfy.ref_token(0)} sits at the round table in the library "
                f"from {kcomfy.ref_token(1)}, reading a book, medium shot.", 4, ref_names)
assert took < MAX_SECONDS, f"reference picture took {took:.0f}s, limit {MAX_SECONDS}s: too slow for the trial"
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Basic auth is enforced by kcomfy's reverse proxy on :8189, so it does not depend on the tunnel's plan
# (ngrok's basic-auth Traffic Policy action is not included in its free plan). The tunnel points at the proxy.
import re, threading, urllib.request

proxy = kcomfy.serve_auth_proxy(8189, "http://127.0.0.1:8188", session["user"], session["password"])
tunnel_proc = None
if TUNNEL == "ngrok":
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    DOMAIN = session.get("domain")
    if DOMAIN:
        # The account's fixed dev domain: the local side knows the address before this cell runs. pyngrok (v3 config,
        # its default) takes it as `url=`. No fallback to a random address: the local side would wait on the wrong one.
        try:
            PUBLIC = ngrok.connect(addr="8189", url="https://" + DOMAIN).public_url.replace("http://", "https://", 1)
        except Exception as e:
            raise RuntimeError(f"ngrok could not open the tunnel on {DOMAIN}: {type(e).__name__}: {e}") from None
        assert PUBLIC == "https://" + DOMAIN, f"ngrok opened {PUBLIC}, not https://{DOMAIN}"
    else:
        PUBLIC = ngrok.connect("8189", "http").public_url.replace("http://", "https://", 1)
else:
    CF = Path('/tmp/cloudflared')
    if not CF.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64", CF)
        CF.chmod(0o755)
    tunnel_proc = subprocess.Popen([str(CF), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8189"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    PUBLIC, t_end = None, time.time() + 120
    while PUBLIC is None and time.time() < t_end:
        line = tunnel_proc.stdout.readline()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
        if m:
            PUBLIC = m.group(0)
    assert PUBLIC, "cloudflared printed no trycloudflare.com URL in 2 min"
    # keep draining its output so the pipe never fills
    threading.Thread(target=lambda: [None for _ in tunnel_proc.stdout], daemon=True).start()

# Check the whole path once, from outside in: tunnel -> proxy (auth) -> ComfyUI.
remote = kcomfy.Comfy(PUBLIC, session["user"], session["password"], timeout=30, retries=6)
remote.ping()
try:
    kcomfy.Comfy(PUBLIC, timeout=30, retries=2).ping()
    raise AssertionError("the tunnel answered without a password")
except kcomfy.ComfyError as e:
    assert "401" in str(e), e
print("KTRIAL URL " + PUBLIC, flush=True)

In [ ]:
# Keep alive until the stop file, the idle limit or the hours cap. Then stop everything and keep the log.
STOP = COMFY / 'input' / kcomfy.STOP_NAME
IDLE_MIN = float(session.get("idle_minutes", 20))
MAX_H = float(session.get("max_hours", 6))
started = last_active = time.time()
last_beat, seen_hist, reason = 0.0, None, None
try:
    while reason is None:
        now = time.time()
        try:
            busy = local.queue_size()
            hist = local.history_count()
        except kcomfy.ComfyError as e:
            print("KTRIAL ComfyUI not answering:", e, flush=True)
            if comfy_proc.poll() is not None:
                reason = f"ComfyUI exited with {comfy_proc.returncode}"
                break
            busy, hist = 0, seen_hist
        if busy or hist != seen_hist:
            last_active = now
        seen_hist = hist
        last_active = max(last_active, proxy.last_request)
        idle = (now - last_active) / 60
        if STOP.exists():
            reason = "stop file"
        elif idle >= IDLE_MIN:
            reason = f"idle {idle:.0f} min"
        elif now - started >= MAX_H * 3600:
            reason = f"max {MAX_H:g} h"
        elif now - last_beat >= 300:
            print(f"KTRIAL alive, idle {idle:.0f} min, done {hist}", flush=True)
            last_beat = now
        if reason is None:
            time.sleep(30)
finally:
    print("KTRIAL stopping:", reason, flush=True)
    try:
        if TUNNEL == "ngrok":
            ngrok.kill()
        elif tunnel_proc is not None:
            tunnel_proc.terminate()
    except Exception as e:
        print("tunnel stop:", e)
    proxy.shutdown()
    comfy_proc.terminate()
    try:
        comfy_proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        comfy_proc.kill()
    log_f.close()
    shutil.copy(LOG, '/kaggle/working/comfy.log')
    print("KTRIAL stopped", flush=True)